# 05 Create SFINCS Scenarios

## Stage Contract

Requires:
- Event Catalog scenario rows
- Wflow base models with prepared native instates/restart files
- SFINCS base model domains with native infiltration, roughness, and source points

## Imports and Runtime


In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import yaml

location_root = Path("..").resolve()
repo_root = location_root.parents[1]
src_root = repo_root / "src"
sys.path.insert(0, str(src_root))

from sfincs_runs.scenarios import handoff_readiness
from wflow_runs.notebook import load_runtime

runtime = load_runtime(location_root, wflow_domain_review_required=False)
location_root = runtime.location_root
location_name = runtime.location_name
runtime_config = runtime.runtime_config
config = runtime.config
sfincs_config = runtime.sfincs_config
wflow_config = runtime.wflow_config
wflow_handoff_manifest = runtime.wflow_handoff_manifest
resolve_location_path = runtime.resolve_location_path

pd.set_option("display.max_colwidth", 140)


## Rerun Control


In [ ]:
rerun = False


## Required Inputs


In [ ]:
from wflow_runs.notebook import exists_table
exists_table(location_root, {
    "scenario catalog": "data/event_catalog/catalog/scenario_catalog.csv",
    "probability catalog": "data/event_catalog/catalog/probability_catalog.csv",
    "resilience stress/training catalog": "data/event_catalog/catalog/resilience_stress_training_catalog.csv",
    "inland design catalog": "data/event_catalog/catalog/inland_design_event_catalog.csv",
    "wflow handoff manifest": wflow_handoff_manifest,
    "wflow events root": wflow_config["wflow"]["events_root"],
    "sfincs domain manifest": sfincs_config["sfincs_domain_set"]["domain_manifest"],
    "sfincs base model root": sfincs_config["paths"]["base_model_root"],
})


## Scenario Build Plan


In [3]:
scenario_build = sfincs_config["scenario_build"]
pd.Series({
    "design_scenario": scenario_build["design_scenario"],
    "forcing_variable": scenario_build["forcing_variable"],
    "zsini_mode": scenario_build["zsini_mode"],
    "spinup_hours": scenario_build["timing"]["spinup_hours"],
    "drain_down_hours": scenario_build["timing"]["drain_down_hours"],
    "min_run_hours": scenario_build["timing"]["min_run_hours"],
    "max_run_hours": scenario_build["timing"]["max_run_hours"],
})


design_scenario     base
forcing_variable    auto
zsini_mode           dry
spinup_hours          12
drain_down_hours      24
min_run_hours         72
max_run_hours        168
dtype: object

## Wflow Event Forcing


In [4]:
pd.Series({
    "discharge_source": runtime_config["inland_coupling"]["discharge_forcing"]["source"],
    "events_root": wflow_config["wflow"]["events_root"],
    "handoff_manifest": wflow_handoff_manifest,
    "acceptance_artifact": "data/wflow/events/<event_id>/sfincs_discharge.dynamic_handoff.json",
    "sfincs_discharge": "data/wflow/events/<event_id>/sfincs_discharge.nc",
    "zero_rain_control_required": True,
}, name="dynamic_wflow_handoff_contract")


discharge_source                                                                   wflow_dynamic
events_root                                                                    data/wflow/events
handoff_manifest                                              data/wflow/domain_set_handoff.yaml
acceptance_artifact           data/wflow/events/<event_id>/sfincs_discharge.dynamic_handoff.json
sfincs_discharge                                data/wflow/events/<event_id>/sfincs_discharge.nc
zero_rain_control_required                                                                  True
Name: dynamic_wflow_handoff_contract, dtype: object

## Coupled Wflow-SFINCS Cluster Pipeline Prep


In [5]:
# This prepares the cluster worklist and commands for the event-atomic production path:
# Wflow dynamic handoff -> native SFINCS staging -> SFINCS run, per event.
# It does not run Wflow or SFINCS locally.
coupled_pipeline_slurm = "cluster/run_wflow_sfincs_dsai_inland_coupled.slurm"
scenario_root_for_worklists = runtime.sfincs_scenarios_root
scenario_root_for_worklists.mkdir(parents=True, exist_ok=True)

pipeline_readiness = handoff_readiness(
    runtime_config,
    location_root,
    catalog_path=resolve_location_path("data/event_catalog/catalog/scenario_catalog.csv"),
    limit=scenario_limit if "scenario_limit" in globals() else None,
)
pipeline_accepted = pipeline_readiness.loc[pipeline_readiness["status"].eq("accepted")].copy()
pipeline_blocked = pipeline_readiness.loc[pipeline_readiness["status"].eq("blocked")].copy()
pipeline_incompatible = pipeline_readiness.loc[pipeline_readiness["status"].eq("incompatible")].copy()
pipeline_joint = pipeline_readiness.loc[pipeline_readiness["status"].isin(["accepted", "blocked"])].copy()

joint_worklist_csv = runtime.joint_worklist_path
readiness_worklist_csv = runtime.readiness_path
blocked_worklist_csv = runtime.blocked_path
accepted_worklist_csv = runtime.accepted_path
incompatible_worklist_csv = runtime.incompatible_path
pipeline_joint.to_csv(joint_worklist_csv, index=False)
pipeline_readiness.to_csv(readiness_worklist_csv, index=False)
pipeline_blocked.to_csv(blocked_worklist_csv, index=False)
pipeline_accepted.to_csv(accepted_worklist_csv, index=False)
pipeline_incompatible.to_csv(incompatible_worklist_csv, index=False)

coupled_array_chunks = 32
coupled_max_concurrent_nodes = 8
# Pick a real catalog event so the debug command below is copy-pasteable.
coupled_debug_event = (
    str(pipeline_joint["event_id"].iloc[0])
    if not pipeline_joint.empty
    else "<event_id>"
)

rerun_flag = int(rerun)

coupled_pipeline_commands = [
    f"SYNC_LOCATION={location_name} cluster/sync_to_dsai.sh --run-inputs-only",
    "mkdir -p runs",
    f"sbatch --array=0-0 --export=ALL,LOCATION={location_name},EVENT_IDS={coupled_debug_event},FORCE_RERUN={rerun_flag},FORCE_WFLOW={rerun_flag},OVERWRITE_METEO={rerun_flag},KEEP_STAGE=1 {coupled_pipeline_slurm}",
    f"sbatch --array=0-{coupled_array_chunks - 1}%{coupled_max_concurrent_nodes} --export=ALL,LOCATION={location_name},WORKLIST=locations/{location_name}/data/sfincs/scenarios/{location_name}_joint_wflow_sfincs_worklist.csv,FORCE_RERUN={rerun_flag},FORCE_WFLOW={rerun_flag},OVERWRITE_METEO={rerun_flag} {coupled_pipeline_slurm}",
]
print("# Submit this joint pipeline after syncing inputs to the cluster:")
for command in coupled_pipeline_commands:
    print(command)

cluster_plan = {
    "location": location_name,
    "pipeline": "wflow_dynamic_handoff_then_sfincs",
    "events_in_joint_worklist": int(len(pipeline_joint)),
    "accepted_before_cluster_run": int(len(pipeline_accepted)),
    "events_without_current_handoff": int(len(pipeline_blocked)),
    "incompatible_events_excluded": int(len(pipeline_incompatible)),
    "joint_worklist_csv": str(joint_worklist_csv),
    "readiness_worklist_csv": str(readiness_worklist_csv),
    "blocked_worklist_csv": str(blocked_worklist_csv),
    "accepted_worklist_csv": str(accepted_worklist_csv),
    "incompatible_worklist_csv": str(incompatible_worklist_csv),
    "coupled_pipeline_slurm": str(repo_root / coupled_pipeline_slurm),
    "rerun": rerun,
    "overwrite_meteo": True,
    "commands": coupled_pipeline_commands,
}
cluster_plan_path = scenario_root_for_worklists / f"{location_name}_joint_wflow_sfincs_cluster_plan.json"
cluster_plan_path.write_text(json.dumps(cluster_plan, indent=2) + "\n", encoding="utf-8")

display(pd.Series(cluster_plan, name="coupled_wflow_sfincs_cluster_pipeline"))


# Submit this joint pipeline after syncing inputs to the cluster:
SYNC_LOCATION=austin cluster/sync_to_dsai.sh --run-inputs-only
mkdir -p runs
sbatch --array=0-0 --export=ALL,LOCATION=austin,EVENT_IDS=design_0000,FORCE_RERUN=1,FORCE_WFLOW=1,OVERWRITE_METEO=1,KEEP_STAGE=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm
sbatch --array=0-31%8 --export=ALL,LOCATION=austin,WORKLIST=locations/austin/data/sfincs/scenarios/austin_joint_wflow_sfincs_worklist.csv,FORCE_RERUN=1,FORCE_WFLOW=1,OVERWRITE_METEO=1 cluster/run_wflow_sfincs_dsai_inland_coupled.slurm


location                                                                                                                                                               austin
pipeline                                                                                                                                    wflow_dynamic_handoff_then_sfincs
events_in_joint_worklist                                                                                                                                                  507
accepted_before_cluster_run                                                                                                                                                 1
events_without_current_handoff                                                                                                                                            506
incompatible_events_excluded                                                                                                      